The user asks a question. Relevant information is retrieved from the vector database and passed to the LLM as context, which then generates the answer.

In [2]:
# pip install langchain-groq python-dotenv

In [3]:
from langchain_huggingface import HuggingFaceEmbeddings
from langchain_community.vectorstores import FAISS

embedding_model = HuggingFaceEmbeddings(
    model_name="sentence-transformers/all-MiniLM-L6-v2"
)
vector_store = FAISS.load_local(
    "../data/faiss_index", embedding_model,
    allow_dangerous_deserialization=True
)
print("Loaded chunks:", vector_store.index.ntotal)


c:\Users\sahil\AppData\Local\Programs\Python\Python313\Lib\site-packages\requests\__init__.py:113: RequestsDependencyWarning: urllib3 (2.6.2) or chardet (7.4.3)/charset_normalizer (3.4.4) doesn't match a supported version!
  warnings.warn(
C:\Users\sahil\AppData\Local\Temp\ipykernel_31832\1720079251.py:2: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.
  from langchain_community.vectorstores import FAISS
W1119 15:02:27.297000 31832 site-packages\torch\distributed\elastic\multiprocessing\redirects.py:29] NOTE: Redirects are currently not supported in Windows or MacOs.


Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

Loaded chunks: 144


In [6]:
from dotenv import load_dotenv
from langchain_groq import ChatGroq

load_dotenv("../.env")

llm = ChatGroq(model="openai/gpt-oss-120b", temperature=0)


In [10]:
def ask(question):
    docs = vector_store.similarity_search(question, k=3)
    context = "\n\n".join(d.page_content for d in docs)

    prompt = f"""Answer the question using ONLY the context below.
If the answer is not in the context, say exactly: I don't have this information in the documents.

Context:
{context}

Question: {question}

Answer:"""

    response = llm.invoke(prompt)
    print(response.content)
    print("\n--- Sources ---")
    for d in docs:
        print("-", d.metadata["title"], "|", d.metadata["source"])

ask("who is a ceo of notion?")


I don't have this information in the documents.

--- Sources ---
- Change your Notion plan | https://www.notion.com/help/upgrade-or-downgrade-your-plan
- Change your Notion plan | https://www.notion.com/help/upgrade-or-downgrade-your-plan
- What is Notion AI? | https://www.notion.com/help/notion-ai-faqs
